# Amazon Nova Forge SDK - Serverless Multi-Turn RL Training (MTRL) Quick Start

This notebook provides a walkthrough of the Amazon Nova Forge SDK for multi-turn reinforcement fine-tuning (MTRL) using SMTJ Serverless with a Bedrock AgentCore agent.

## What You'll Learn

1. Configuring a Bedrock AgentCore runtime as the agent environment
2. Launching an MTRL training job with hyperparameter overrides
3. Monitoring training progress, viewing metrics
4. Saving/loading job results across sessions
5. Running MTRL evaluation (base model, fine-tuned, or comparison)
6. Iterative training (SFT → MTRL)
7. Deploying to SageMaker and Bedrock

## Table of Contents
- [Step 1: Import Required Modules](#step-1-import-required-modules)
- [Step 2: Configure Your AWS Resources](#step-2-configure-your-aws-resources)
- [Step 3: Configure Runtime Infrastructure](#step-3-configure-runtime-infrastructure)
- [Step 4: Initialize ForgeTrainer](#step-4-initialize-forgetrainer)
- [Step 5: Start Training](#step-5-start-training)
- [Step 6: Monitor Training Progress](#step-6-monitor-training-progress)
- [Step 7: Save and Load Job Results](#step-7-save-and-load-job-results)
- [Step 8: Evaluate Your Model](#step-8-evaluate-your-model-after-training-completes)
- [Step 9: Evaluation Modes](#step-9-evaluation-modes)
- [Step 10: Iterative Training (SFT → MTRL)](#step-10-iterative-training-sft--mtrl)
- [Step 11: Deploy Your Model](#step-11-deploy-your-model)

## Prerequisites

- AWS credentials configured
- A Bedrock AgentCore runtime deployed
- S3 bucket with training prompts (parquet format)
- IAM execution role with SageMaker permissions
- Nova Forge SDK installed per [its README](https://github.com/aws/nova-forge-sdk/blob/main/README.md#installation)

## Step 1: Import Required Modules

In [ ]:
!pip install amzn-nova-forge

In [ ]:
import os

import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound


def load_credentials(profile=None):
    """Load AWS credentials with fallback behavior."""
    if profile:
        try:
            session = boto3.Session(profile_name=profile)
            credentials = session.get_credentials()
            if not credentials:
                raise RuntimeError(f"No credentials found for profile '{profile}'")
        except ProfileNotFound:
            raise RuntimeError(f"Profile '{profile}' not found in credentials file")
    else:
        try:
            session = boto3.Session()
            credentials = session.get_credentials()
            if not credentials:
                raise RuntimeError("No credentials found in current AWS session")
        except NoCredentialsError:
            raise RuntimeError("No AWS credentials configured")

    # Validate credentials
    try:
        sts_client = session.client("sts")
        sts_client.get_caller_identity()
    except ClientError as e:
        raise RuntimeError(f"Invalid AWS credentials: {e}")

    return {
        "aws_access_key_id": credentials.access_key,
        "aws_secret_access_key": credentials.secret_key,
        "aws_session_token": credentials.token,
        "region_name": session.region_name or "us-east-1",
    }

In [ ]:
load_credentials()
print("AWS credentials validated successfully!")

In [ ]:
from amzn_nova_forge import *
from amzn_nova_forge.core.result import BaseJobResult

print("SDK imported successfully!")

## Step 2: Configure Your AWS Resources

In [ ]:
# TODO: Update these values for your environment
ACCOUNT_ID = "<your-account-id>"
REGION = "us-east-1"

EXECUTION_ROLE = f"arn:aws:iam::{ACCOUNT_ID}:role/<your-execution-role>"
MODEL_PKG_GROUP = "<your-model-package-group-name>"
MLFLOW_ARN = f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:mlflow-tracking-server/<your-mlflow-server>"

# Agent environment: Bedrock AgentCore runtime ARN
AGENT_CORE_ARN = f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:runtime/<your-agent-runtime>"

# Training and evaluation data
TRAINING_DATA = f"s3://<your-bucket>/prompts/train.parquet"
EVAL_DATA = f"s3://<your-bucket>/prompts/eval.parquet"

# Output path
OUTPUT_S3 = f"s3://<your-bucket>/output/"

print(f"Training Data: {TRAINING_DATA}")
print(f"Output Path:   {OUTPUT_S3}")

## Step 3: Configure Runtime Infrastructure

In [ ]:
# For MTRL, use agent_core_arn instead of rft_lambda
runtime = SMTJServerlessRuntimeManager(
    model_package_group_name=MODEL_PKG_GROUP,
    execution_role=EXECUTION_ROLE,
    agent_core_arn=AGENT_CORE_ARN,
)

print("Runtime configured for SMTJ Serverless (MTRL)")

## Step 4: Initialize ForgeTrainer

In [ ]:
# Create MLflow monitor (optional)
mlflow_monitor = MLflowMonitor(tracking_uri=MLFLOW_ARN)

# Create trainer
trainer = ForgeTrainer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.RFT_MULTITURN_LORA,
    infra=runtime,
    training_data_s3_path=TRAINING_DATA,
    config=ForgeConfig(
        output_s3_path=OUTPUT_S3,
        mlflow_monitor=mlflow_monitor,
    ),
    region=REGION,
)

print("ForgeTrainer initialized")
print(f"   Model: Nova Lite 2.0")
print(f"   Method: RFT Multi-Turn with LoRA")

## Step 5: Start Training

Overrides are applied directly to the `MultiTurnRLTrainer.hyperparameters` object. Use exact parameter names.

In [ ]:
# Define training hyperparameters
training_overrides = {
    "global_batch_size": 16,
    "max_steps": 50,
    "rollout_timeout": 600,
    # Other available overrides:
    # "learning_rate": 4e-5,
    # "lora_rank": 32,
    # "lora_alpha": 64,
    # "advantage_method": "group_based",
    # "group_size": 8,
    # "rollout_max_concurrency": 96,
    # "sampling_temperature": 0.7,
    # "top_p": 0.95,
    # "save_every": 10,
    # "eval_every": 10,
}

# Start training
training_result = trainer.train(
    job_name="<your-job-name>",
    overrides=training_overrides,
)

print("\nTraining job started!")
print(f"   Job ID:      {training_result.job_id}")
print(f"   Output Path: {training_result.model_artifacts.output_s3_path}")

## Step 6: Monitor Training Progress

In [ ]:
# Block until the job reaches a terminal state (displays rich progress panel)
training_result.wait(poll=30, timeout=7200)

### Check Job Status and Metrics

In [ ]:
status, raw_status = training_result.get_job_status()
print(f"Status:               {status.value} ({raw_status})")
print(f"Output Model Package: {training_result.model_artifacts.output_model_package_arn}")

In [ ]:
# View per-step training metrics from MLflow
training_result.get_training_metrics()

## Step 7: Save and Load Job Results

In [ ]:
training_result_path = training_result.dump()
print("Training result saved")

In [ ]:
loaded_result = TrainingResult.load(training_result_path)
print(f"Loaded - Job ID: {loaded_result.job_id}")

# All MTRL operations work on the loaded result
status, raw = loaded_result.get_job_status()
print(f"Status: {status.value} ({raw})")
print(f"Output Model Package: {loaded_result.model_artifacts.output_model_package_arn}")

## Step 8: Evaluate Your Model (After Training Completes)

Run MTRL evaluation on the trained model using the same agent environment.

In [ ]:
evaluator = ForgeEvaluator(
    model=Model.NOVA_LITE_2,
    infra=runtime,
    data_s3_path=EVAL_DATA,
    config=ForgeConfig(
        output_s3_path=OUTPUT_S3,
        mlflow_monitor=mlflow_monitor,
    ),
    region=REGION,
)

In [ ]:
# Pass training_result to auto-resolve the model checkpoint
eval_result = evaluator.evaluate(
    job_name="<your-eval-job-name>",
    eval_task=EvaluationTask.RFT_MULTITURN_EVAL,
    job_result=training_result,
    overrides={
        "global_batch_size": 16,
        "max_steps": 20,
    },
)

print(f"Eval Job ID:  {eval_result.job_id}")
print(f"Eval Output:  {eval_result.eval_output_path}")

## Step 9: Evaluation Modes

You can evaluate the base model, fine-tuned model, or compare both in a single pipeline.

In [ ]:
# Fine-tuned model only (pass model_path directly)
eval_finetuned = evaluator.evaluate(
    job_name="eval-finetuned",
    eval_task=EvaluationTask.RFT_MULTITURN_EVAL,
    model_path=training_result.model_artifacts.output_model_arn,
)

# Base + fine-tuned comparison (both in one pipeline)
eval_comparison = evaluator.evaluate(
    job_name="eval-comparison",
    eval_task=EvaluationTask.RFT_MULTITURN_EVAL,
    model_path=training_result.model_artifacts.output_model_arn,
    task_config=EvalTaskConfig(evaluate_base_model=True),
)
print(f"Comparison eval: {eval_comparison.job_id}")

## Step 10: Iterative Training (SFT → MTRL)

Use an SFT checkpoint as the starting point for MTRL. Register the checkpoint in a model package group, then pass it as `model_arn` to `ForgeTrainer`.

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=REGION)

# 1. Get SFT checkpoint path from manifest (after SFT job completes)
SFT_CHECKPOINT = "s3://customer-escrow-<account>-smtj-<id>/<sft-job-name>/step_10"

# 2. Register in a model package group
resp = sm.create_model_package(
    ModelPackageGroupName="<your-sft-mpg>",
    InferenceSpecification={
        "Containers": [
            {
                "ModelDataSource": {
                    "S3DataSource": {
                        "S3Uri": SFT_CHECKPOINT,
                        "S3DataType": "S3Prefix",
                        "CompressionType": "None",
                    }
                },
                "IsCheckpoint": False,
                "BaseModel": {
                    "HubContentName": "nova-textgeneration-lite-v2",
                    "HubContentVersion": "3.48.0",
                },
            }
        ],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
    },
    SkipModelValidation="All",
)
SFT_MODEL_PACKAGE_ARN = resp["ModelPackageArn"]
print(f"SFT model package: {SFT_MODEL_PACKAGE_ARN}")

# 3. Launch MTRL with SFT checkpoint as base
trainer_iterative = ForgeTrainer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.RFT_MULTITURN_LORA,
    infra=runtime,
    training_data_s3_path=TRAINING_DATA,
    model_arn=SFT_MODEL_PACKAGE_ARN,  # SFT checkpoint as starting point
    config=ForgeConfig(output_s3_path=OUTPUT_S3, mlflow_monitor=mlflow_monitor),
    region=REGION,
)

iterative_result = trainer_iterative.train(
    job_name="sft-to-mtrl",
    overrides={"global_batch_size": 8, "max_steps": 12, "save_every": 10},
)
print(f"SFT→MTRL Job: {iterative_result.job_id}")

## Step 11: Deploy Your Model

Deploy the trained model to SageMaker or Bedrock for inference.

In [ ]:
MODEL_PACKAGE_ARN = training_result.model_artifacts.output_model_arn

deployer = ForgeDeployer(region=REGION, model=Model.NOVA_LITE_2)

# Option A: Deploy to SageMaker endpoint
smi_result = deployer.deploy(
    model_artifact_path=MODEL_PACKAGE_ARN,
    deploy_platform=DeployPlatform.SAGEMAKER,
    sagemaker_instance_type="ml.g6.48xlarge",
    endpoint_name="my-mtrl-endpoint",
    execution_role_name="<your-execution-role>",
)
print(f"SageMaker endpoint: {smi_result.endpoint.uri}")

# Option B: Deploy to Bedrock On-Demand
bedrock_model = deployer.create_custom_model(
    model_artifact_path=None,
    custom_model_data_source={"modelPackageArnDataSource": {"modelPackageArn": MODEL_PACKAGE_ARN}},
    endpoint_name="my-mtrl-bedrock",
    execution_role_name="<your-execution-role>",
)
bedrock_result = deployer.deploy_to_bedrock(
    model_deploy_result=bedrock_model,
    deploy_platform=DeployPlatform.BEDROCK_OD,
)
print(f"Bedrock deployment: {bedrock_result.endpoint.uri}")

In [ ]:
# Invoke the deployed model
inference = ForgeInference(region=REGION)

result = inference.invoke(
    endpoint_arn=smi_result.endpoint.uri,  # or bedrock_result.endpoint.uri
    request_body={
        "messages": [{"role": "user", "content": "What is 25 * 4 + 10?"}],
        "max_tokens": 256,
    },
)
result.show()

---
## Summary

| What | How |
|------|-----|
| Launch training | `trainer.train(job_name=..., overrides={...})` |
| Wait for completion | `training_result.wait()` |
| Check status | `training_result.get_job_status()` |
| View metrics | `training_result.get_training_metrics()` |
| Get model package | `training_result.model_artifacts.output_model_arn` |
| Save/load result | `training_result.dump()` / `TrainingResult.load(path)` |
| Eval (fine-tuned) | `evaluator.evaluate(model_path=arn, ...)` |
| Eval (base + FT) | `evaluator.evaluate(model_path=arn, task_config=EvalTaskConfig(evaluate_base_model=True))` |
| SFT → MTRL | Register SFT checkpoint in MPG, pass as `model_arn` to `ForgeTrainer` |
| Deploy to SMI | `deployer.deploy(model_artifact_path=arn, deploy_platform=DeployPlatform.SAGEMAKER)` |
| Deploy to Bedrock | `deployer.create_custom_model(custom_model_data_source=...) + deploy_to_bedrock()` |
| Invoke | `ForgeInference(region).invoke(endpoint_arn=..., request_body={...})` |